# Lab 03 · AQE and how operators run

**Part A** (about 45 min): see how `avg`, `count_distinct`, sorting and AQE run, in plans and in the Spark UI.<br>
**Part B** (about 75 min): 13 problems with window functions: ranks, neighbours, running totals.

**Checklist questions Part A gives you evidence for:**<br>
<span style="color:#d93025"><b>1.4</b> Adaptive Query Execution (AQE) · <b>1.9</b> count() vs. approx_count_distinct() · <b>1.14</b> How count(distinct) works · <b>1.15</b> How avg() works · <b>1.16</b> How sorting works · <b>3.11</b> spark.sql.shuffle.partitions</span>

## How to work
* Run the **Setup** cell first, then answer the questions in order. `F` is `pyspark.sql.functions`, `Window` is `pyspark.sql.Window`.
* **Part A**: run the code under the question (or add a line), look at the output or the Spark UI, and write your answer as a comment.
* **Part B**: write the code yourself. Your result must look exactly like the **expected output**, with the rows in the same order.
* The answers are collapsed at the very end of the notebook. Open one only after you have tried.

## Documentation
* [Adaptive Query Execution](https://spark.apache.org/docs/latest/sql-performance-tuning.html#adaptive-query-execution): what AQE changes at runtime
* [Coalescing post-shuffle partitions](https://spark.apache.org/docs/latest/sql-performance-tuning.html#coalescing-post-shuffle-partitions): AQE and `spark.sql.shuffle.partitions`
* [EXPLAIN](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-explain.html): the `formatted` mode shows each operator's inputs and outputs
* [approx_count_distinct](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.approx_count_distinct.html) and [HyperLogLog](https://en.wikipedia.org/wiki/HyperLogLog)
* [ORDER BY](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-orderby.html): global sort
* [Window functions](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-window.html): ranking, analytic and aggregate functions over a window
* [Window (PySpark)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/window.html): `partitionBy`, `orderBy`, `rowsBetween`
* [Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html): `row_number`, `rank`, `dense_rank`, `lag`, `lead`, `sum`, `avg`

## The data
Three small tables of an online shop. The setup cell creates them, and you can read every row in it.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date* (the day the customer registered)
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)

Good to know:
* order 1011 is a guest checkout: it has no *customer_id*
* customer 6 has no email and no city
* customer 7 has never ordered, and product 107 has never been sold
* order 1016 is for product 108, which is no longer in *products*

These are the same tables as in Labs 01 and 02.

## Setup
Run this cell once. It starts Spark, saves the three tables as Parquet files in `data/shop/` and loads them into `customers`, `products` and `orders`.

In [ ]:
# Setup: starts Spark and creates the shop tables. Run it once.
import logging, os, shutil, sys, time
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder
         .master("local[4]")   # 4 tasks at a time on your machine
         .appName("lab03")
         .config("spark.sql.adaptive.enabled", "false")        # AQE starts off; question A7 turns it on
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")                  # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers_rows = [
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
]
products_rows = [
    # product_id, product_name, category, price
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
]
orders_rows = [
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
]

# Write the tables as Parquet files and read them back, the way real data arrives.
# data/shop belongs to this lab: it is rebuilt every time you run this cell.
shutil.rmtree("data/shop", ignore_errors=True)
for table, rows, schema in [
    ("customers", customers_rows, "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE"),
    ("products",  products_rows,  "product_id INT, product_name STRING, category STRING, price DOUBLE"),
    ("orders",    orders_rows,    "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING"),
]:
    spark.createDataFrame(rows, schema).coalesce(1).write.parquet(f"data/shop/{table}")

customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")

def count_jobs(action):
    """Runs action() and returns how many Spark jobs it started."""
    tracker = spark.sparkContext.statusTracker()
    before = set(tracker.getJobIdsForGroup())
    action()
    time.sleep(0.5)                       # the Spark UI records a job a moment after it ends
    return len(set(tracker.getJobIdsForGroup()) - before)

print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

Look at the three tables:

In [ ]:
customers.show()
products.show()
orders.show()

## Part A · How operators run, and what AQE changes

### How aggregates run <span style="color:#d93025">(1.15, 1.14, 1.9)</span>

**A1. Print the plan of `orders.groupBy("status").agg(F.avg("quantity"))` with `explain("formatted")`. What does the first HashAggregate keep for each status: an average, or something else?**

In [ ]:
# My answer:
orders.groupBy("status").agg(F.avg("quantity")).explain("formatted")

**A2. Why can't Spark simply average the partial averages? Compare the average quantity of all orders with the average of the customers' average quantities.**

In [ ]:
# My answer:
per_customer = (orders.filter(F.col("customer_id").isNotNull())
                      .groupBy("customer_id").agg(F.avg("quantity").alias("avg_quantity")))

**A3. How many `HashAggregate` and `Exchange` nodes does the plan of `orders.agg(F.count_distinct("customer_id"))` have? Compare with `orders.agg(F.count("customer_id"))`.**

In [ ]:
# My answer:
orders.agg(F.count_distinct("customer_id")).explain()

**A4. `approx_count_distinct` estimates instead of counting exactly. On 10 million visits by 1,000,000 visitors, how far off is it, and how much faster?**

In [ ]:
# My answer:
visits = spark.range(10_000_000).select((F.col("id") % 1_000_000).alias("visitor"))   # 1,000,000 different visitors
start = time.time()
print("exact: ", visits.agg(F.count_distinct("visitor")).first()[0], f"{time.time() - start:.2f} s")

### How sorting runs <span style="color:#d93025">(1.16)</span>

**A5. Sorting all orders with `orderBy` starts more than one job. Guess how many, then measure it with `count_jobs`. Which partitioning does the `Exchange` in the plan use?**

In [ ]:
# My guess:
sorted_orders = orders.orderBy("quantity")

**A6. What does `limit` change? Look at the plan of the 3 orders with the largest quantity.**

In [ ]:
# My answer:
orders.orderBy(F.desc("quantity")).limit(3).explain()

### AQE and spark.sql.shuffle.partitions <span style="color:#d93025">(1.4, 3.11)</span>

**A7. Turn AQE on and collect `q` (cell below). In `q.explain()`, what did AQE do with the 200 shuffle partitions?**

In [ ]:
# My answer:
spark.conf.set("spark.sql.adaptive.enabled", "true")
q = orders.groupBy("customer_id").count()
q.collect()

**A8. Keep AQE on, set `spark.sql.shuffle.partitions` to 2000, build `q` again and collect it. How many tasks does the last stage have now (Spark UI, newest job)?**

In [ ]:
# My answer:
spark.conf.set("spark.sql.shuffle.partitions", 2000)

**Done with Part A?** Explain each red checklist question at the top aloud, in your own words, using what you saw.

## Part B · Window functions
13 problems. A window function computes a value for every row from a group of related rows (its window), without collapsing the rows like `groupBy` does. Break ties exactly as the question says, so your rows match the expected output.

**B1. Number the orders of customers 1 and 2 by date: show customer_id, order_id, order_date and order_no (1 = the customer's first order). On the same day, the smaller order_id comes first.**

Expected output:

```
+-----------+--------+----------+--------+
|customer_id|order_id|order_date|order_no|
+-----------+--------+----------+--------+
|          1|    1001|2024-01-10|       1|
|          1|    1002|2024-01-10|       2|
|          1|    1007|2024-02-20|       3|
|          1|    1015|2024-03-25|       4|
|          2|    1003|2024-01-25|       1|
|          2|    1005|2024-02-05|       2|
|          2|    1013|2024-03-18|       3|
+-----------+--------+----------+--------+
```

<details><summary>Hint</summary>

`w = Window.partitionBy("customer_id").orderBy("order_date", "order_id")`, then `F.row_number().over(w)`.

</details>

**B2. Rank the products by price inside each category, most expensive first. Show category, product_name, price and price_rank.**

Expected output:

```
+-----------+------------+------+----------+
|   category|product_name| price|price_rank|
+-----------+------------+------+----------+
|      Books|    Cookbook|  25.0|         1|
|      Books|       Novel|  15.0|         2|
|Electronics|      Laptop|1200.0|         1|
|Electronics|  Headphones| 150.0|         2|
|       Home|   Desk Lamp|  40.0|         1|
|    Kitchen|Coffee Maker|  85.0|         1|
|    Kitchen|     Blender|  60.0|         2|
+-----------+------------+------+----------+
```

<details><summary>Hint</summary>

`F.rank()` over a window partitioned by category and ordered by price, descending.

</details>

**B3. Show the most expensive product of each category: category, product_name and price.**

Expected output:

```
+-----------+------------+------+
|   category|product_name| price|
+-----------+------------+------+
|      Books|    Cookbook|  25.0|
|Electronics|      Laptop|1200.0|
|       Home|   Desk Lamp|  40.0|
|    Kitchen|Coffee Maker|  85.0|
+-----------+------------+------+
```

<details><summary>Hint</summary>

Number the rows of each category with `row_number`, then keep number 1. This is the classic top-N per group.

</details>

**B4. For every order of customers 1 and 3, show how many days passed since the same customer's previous order (`days_since_prev`, empty for the first order). Show customer_id, order_id, order_date and days_since_prev.**

Expected output:

```
+-----------+--------+----------+---------------+
|customer_id|order_id|order_date|days_since_prev|
+-----------+--------+----------+---------------+
|          1|    1001|2024-01-10|           NULL|
|          1|    1002|2024-01-10|              0|
|          1|    1007|2024-02-20|             41|
|          1|    1015|2024-03-25|             34|
|          3|    1004|2024-02-03|           NULL|
|          3|    1009|2024-03-05|             31|
|          3|    1016|2024-03-28|             23|
+-----------+--------+----------+---------------+
```

<details><summary>Hint</summary>

`F.lag("order_date").over(w)` is the value of the previous row; `F.datediff(end, start)`.

</details>

**B5. For the orders of customers 2 and 5, show the date of the same customer's next order (`next_order_date`, empty for the last one). Show customer_id, order_id, order_date and next_order_date.**

Expected output:

```
+-----------+--------+----------+---------------+
|customer_id|order_id|order_date|next_order_date|
+-----------+--------+----------+---------------+
|          2|    1003|2024-01-25|     2024-02-05|
|          2|    1005|2024-02-05|     2024-03-18|
|          2|    1013|2024-03-18|           NULL|
|          5|    1008|2024-03-02|     2024-03-20|
|          5|    1014|2024-03-20|           NULL|
+-----------+--------+----------+---------------+
```

<details><summary>Hint</summary>

`F.lead` is the opposite of `F.lag`: the value of the next row.

</details>

**B6. Running total: for customers 1 and 2, show customer_id, order_id, order_date, quantity and `items_so_far` (the items of this order and all earlier orders of the customer).**

Expected output:

```
+-----------+--------+----------+--------+------------+
|customer_id|order_id|order_date|quantity|items_so_far|
+-----------+--------+----------+--------+------------+
|          1|    1001|2024-01-10|       1|           1|
|          1|    1002|2024-01-10|       3|           4|
|          1|    1007|2024-02-20|       1|           5|
|          1|    1015|2024-03-25|       1|           6|
|          2|    1003|2024-01-25|       2|           2|
|          2|    1005|2024-02-05|       1|           3|
|          2|    1013|2024-03-18|       1|           4|
+-----------+--------+----------+--------+------------+
```

<details><summary>Hint</summary>

`F.sum("quantity").over(w)` with `rowsBetween(Window.unboundedPreceding, Window.currentRow)`.

</details>

**B7. For customers 1 and 3, show each order's share of the customer's items: customer_id, order_id, quantity and share_pct (quantity divided by the customer's total quantity, × 100, 1 decimal).**

Expected output:

```
+-----------+--------+--------+---------+
|customer_id|order_id|quantity|share_pct|
+-----------+--------+--------+---------+
|          1|    1001|       1|     16.7|
|          1|    1002|       3|     50.0|
|          1|    1007|       1|     16.7|
|          1|    1015|       1|     16.7|
|          3|    1004|       1|     16.7|
|          3|    1009|       4|     66.7|
|          3|    1016|       1|     16.7|
+-----------+--------+--------+---------+
```

<details><summary>Hint</summary>

A window with `partitionBy` and no `orderBy` covers the whole group: `F.sum(...).over(w)` is the group total.

</details>

**B8. Show the revenue (quantity × price) of delivered orders per month, the previous month's revenue and the change. Columns: month, revenue, prev_revenue, change.**

Expected output:

```
+-----+-------+------------+-------+
|month|revenue|prev_revenue| change|
+-----+-------+------------+-------+
|    1| 1545.0|        NULL|   NULL|
|    2|  175.0|      1545.0|-1370.0|
|    3| 1395.0|       175.0| 1220.0|
+-----+-------+------------+-------+
```

<details><summary>Hint</summary>

Group by month first, then use `F.lag` over `Window.orderBy("month")`.

</details>

**B9. Rank the customers by their number of orders. Customers with the same number share a rank, and no rank is skipped after a tie. Show name, orders and rank, sorted by rank, then name.**

Expected output:

```
+----------------+------+----+
|            name|orders|rank|
+----------------+------+----+
|    Aziz Karimov|     4|   1|
|   Bella Schmidt|     3|   2|
|     Carlos Ruiz|     3|   2|
|Dilnoza Rahimova|     2|   3|
|     Emma Wilson|     2|   3|
|    Farid Aliyev|     1|   4|
+----------------+------+----+
```

<details><summary>Hint</summary>

`F.dense_rank()` gives 1, 2, 2, 3; `F.rank()` would give 1, 2, 2, 4.

</details>

**B10. For each customer, show the name of the product of their first order (the earliest date; on the same day, the smaller order_id). Columns: name and first_product, sorted by name.**

Expected output:

```
+----------------+-------------+
|            name|first_product|
+----------------+-------------+
|    Aziz Karimov|       Laptop|
|   Bella Schmidt|   Headphones|
|     Carlos Ruiz| Coffee Maker|
|Dilnoza Rahimova|      Blender|
|     Emma Wilson|       Laptop|
|    Farid Aliyev| Coffee Maker|
+----------------+-------------+
```

<details><summary>Hint</summary>

Keep each customer's first order with `row_number`, then join customers and products.

</details>

**B11. For customers 1 and 3, show the moving average of quantity over the current and the previous order (`moving_avg`, 2 decimals). Show customer_id, order_id, quantity and moving_avg.**

Expected output:

```
+-----------+--------+--------+----------+
|customer_id|order_id|quantity|moving_avg|
+-----------+--------+--------+----------+
|          1|    1001|       1|       1.0|
|          1|    1002|       3|       2.0|
|          1|    1007|       1|       2.0|
|          1|    1015|       1|       1.0|
|          3|    1004|       1|       1.0|
|          3|    1009|       4|       2.5|
|          3|    1016|       1|       2.5|
+-----------+--------+--------+----------+
```

<details><summary>Hint</summary>

`rowsBetween(-1, Window.currentRow)`: the previous row and this one.

</details>

**B12. Show the orders whose quantity is above the average quantity of the same customer's orders. Columns: customer_id, order_id, quantity and avg_quantity (2 decimals), sorted by order_id.**

Expected output:

```
+-----------+--------+--------+------------+
|customer_id|order_id|quantity|avg_quantity|
+-----------+--------+--------+------------+
|          1|    1002|       3|         1.5|
|          2|    1003|       2|        1.33|
|          3|    1009|       4|         2.0|
|          5|    1014|       5|         3.0|
+-----------+--------+--------+------------+
```

<details><summary>Hint</summary>

Compute the customer's average as a window column, then filter on it (a filter can't use a window function directly).

</details>

**B13. Show the top 2 products of each category by items delivered (ties: the smaller product_id first). Columns: category, product_name, items and rank, sorted by category, then rank.**

Expected output:

```
+-----------+------------+-----+----+
|   category|product_name|items|rank|
+-----------+------------+-----+----+
|      Books|       Novel|   12|   1|
|      Books|    Cookbook|    1|   2|
|Electronics|  Headphones|    3|   1|
|Electronics|      Laptop|    2|   2|
|    Kitchen|     Blender|    1|   1|
+-----------+------------+-----+----+
```

<details><summary>Hint</summary>

Aggregate per product first, then rank inside each category and keep ranks 1 and 2.

</details>

---
**Done?** In the Theory part of Day 3, answer the red checklist questions aloud.

## Answers
Try each question yourself first. Click a question to open its answer; Part A answers also show the output of the reference run.

<details>
<summary><b>A1</b> &nbsp;Print the plan of orders.groupBy("status").agg(F.avg("quantity")) with explain("formatted"). What does the ...</summary>

**A sum and a count** (`partial_avg`, *Aggregate Attributes: [sum, count]*). Every partition adds up the quantities and counts the rows of each status. After the shuffle, the second HashAggregate adds up all the sums and all the counts, and only then divides. That is how `avg` works: it travels as a (sum, count) pair, because averages of parts cannot be combined (A2).

```python
orders.groupBy("status").agg(F.avg("quantity")).explain("formatted")
```

Result:

```
(3) HashAggregate [codegen id : 1]
Input [2]: [quantity#27, status#29]
Keys [1]: [status#29]
Functions [1]: [partial_avg(quantity#27)]
Aggregate Attributes [2]: [sum#89, count#90L]
Results [3]: [status#29, sum#91, count#92L]

(5) HashAggregate [codegen id : 2]
Input [3]: [status#29, sum#91, count#92L]
Keys [1]: [status#29]
Functions [1]: [avg(quantity#27)]
Aggregate Attributes [1]: [avg(quantity#27)#87]
Results [2]: [status#29, avg(quantity#27)#87 AS avg(quantity)#88]
```

</details>

<details>
<summary><b>A2</b> &nbsp;Why can't Spark simply average the partial averages? Compare the average quantity of all orders with the ...</summary>

**The two numbers differ** (1.81 against 1.97). An average of averages gives every customer the same weight, however many orders they placed; the real average gives every order the same weight. Partitions are like the customers here: they hold different numbers of rows, so Spark must merge sums and counts, not averages.

```python
per_customer = (orders.filter(F.col("customer_id").isNotNull())
                      .groupBy("customer_id").agg(F.avg("quantity").alias("avg_quantity")))
orders.agg(F.round(F.avg("quantity"), 2).alias("avg_of_all_orders")).show()
per_customer.agg(F.round(F.avg("avg_quantity"), 2).alias("avg_of_customer_averages")).show()
```

Result:

```
+-----------------+
|avg_of_all_orders|
+-----------------+
|             1.81|
+-----------------+

+------------------------+
|avg_of_customer_averages|
+------------------------+
|                    1.97|
+------------------------+
```

</details>

<details>
<summary><b>A3</b> &nbsp;How many HashAggregate and Exchange nodes does the plan of orders.agg(F.count_distinct("customer_id")) have? ...</summary>

**4 HashAggregates and 2 Exchanges, against 2 and 1 for a plain count.** To count distinct values, Spark first removes the duplicates: it groups by customer_id (partial and final, around a shuffle by customer_id), and only then counts the groups (partial and final, around a second shuffle to one partition). Every distinct value travels through a shuffle, so an exact distinct count of a big column is expensive.

```python
orders.agg(F.count_distinct("customer_id")).explain()
orders.agg(F.count("customer_id")).explain()
```

Result:

```
== Physical Plan ==
*(3) HashAggregate(keys=[], functions=[count(distinct customer_id#25)])
+- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=279]
   +- *(2) HashAggregate(keys=[], functions=[partial_count(distinct customer_id#25)])
      +- *(2) HashAggregate(keys=[customer_id#25], functions=[])
         +- Exchange hashpartitioning(customer_id#25, 200), ENSURE_REQUIREMENTS, [plan_id=274]
            +- *(1) HashAggregate(keys=[customer_id#25], functions=[])
               +- *(1) ColumnarToRow
                  +- FileScan parquet [customer_id#25] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<customer_id:int>


== Physical Plan ==
*(2) HashAggregate(keys=[], functions=[count(customer_id#25)])
+- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=318]
   +- *(1) HashAggregate(keys=[], functions=[partial_count(customer_id#25)])
      +- *(1) ColumnarToRow
         +- FileScan parquet [customer_id#25] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<customer_id:int>
```

</details>

<details>
<summary><b>A4</b> &nbsp;approx_count_distinct estimates instead of counting exactly. On 10 million visits by 1,000,000 visitors, how ...</summary>

**About 7% off, and several times faster** (compare the numbers and times in the result). `approx_count_distinct` uses **HyperLogLog**: each partition builds a small fixed-size sketch from the hashes of its values, and only the sketches are shuffled and merged (one `Exchange SinglePartition` of tiny sketches, instead of a shuffle of every distinct value). The allowed error is a parameter: the default `rsd=0.05` means about 5%; `approx_count_distinct("visitor", 0.01)` is more exact and slower. Not to be confused with `count()`, the action that counts rows exactly.

```python
visits = spark.range(10_000_000).select((F.col("id") % 1_000_000).alias("visitor"))   # 1,000,000 different visitors
start = time.time()
print("exact: ", visits.agg(F.count_distinct("visitor")).first()[0], f"{time.time() - start:.2f} s")
start = time.time()
print("approx:", visits.agg(F.approx_count_distinct("visitor")).first()[0], f"{time.time() - start:.2f} s")
```

Result:

```
exact:  1000000 1.76 s
approx: 925738 0.47 s
```

*Your times will differ.*

</details>

<details>
<summary><b>A5</b> &nbsp;Sorting all orders with orderBy starts more than one job. Guess how many, then measure it with count_jobs. ...</summary>

**2 jobs, and `rangepartitioning`.** A global sort works in three steps: a first job samples the data to choose range boundaries (for example quantity 1, 2 to 3, 4 to 5); then every row is shuffled to the partition of its range; finally each partition is sorted on its own. Partition 1 holds the smallest values, so reading the partitions in order gives a fully sorted result.

```python
sorted_orders = orders.orderBy("quantity")
print(count_jobs(lambda: sorted_orders.collect()), "jobs")
sorted_orders.explain()
```

Result:

```
2 jobs
== Physical Plan ==
*(2) Sort [quantity#27 ASC NULLS FIRST], true, 0
+- Exchange rangepartitioning(quantity#27 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=422]
   +- *(1) ColumnarToRow
      +- FileScan parquet [order_id#24,customer_id#25,product_id#26,quantity#27,order_date#28,status#29] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:int,customer_id:int,product_id:int,quantity:int,order_date:date,status:string>
```

</details>

<details>
<summary><b>A6</b> &nbsp;What does limit change? Look at the plan of the 3 orders with the largest quantity.</summary>

**No global sort any more: `TakeOrderedAndProject`.** Each partition keeps only its own top 3 rows, and those few rows are merged at the end: no range shuffle, and only 1 job. Asking for the top N is much cheaper than sorting everything.

```python
orders.orderBy(F.desc("quantity")).limit(3).explain()
```

Result:

```
== Physical Plan ==
TakeOrderedAndProject(limit=3, orderBy=[quantity#27 DESC NULLS LAST], output=[order_id#24,customer_id#25,product_id#26,quantity#27,order_date#28,status#29])
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#24,customer_id#25,product_id#26,quantity#27,order_date#28,status#29] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:int,customer_id:int,product_id:int,quantity:int,order_date:date,status:string>
```

</details>

<details>
<summary><b>A7</b> &nbsp;Turn AQE on and collect q (cell below). In q.explain(), what did AQE do with the 200 shuffle partitions?</summary>

**It coalesced them** (`AQEShuffleRead coalesced`): after the first stage, AQE saw that the 200 partitions together hold only a few bytes, merged them, and the last stage ran **1 task** instead of 200. This is **Adaptive Query Execution**: at every shuffle, Spark re-plans the rest of the query with the real sizes it just measured. Besides coalescing partitions, AQE switches joins to broadcast joins (Lab 05) and splits skewed partitions (Lab 06). `isFinalPlan=true` marks the plan AQE really ran; the *Initial Plan* is the one made before running.

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
q = orders.groupBy("customer_id").count()
q.collect()
q.explain()
```

Result:

```
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- *(2) HashAggregate(keys=[customer_id#25], functions=[count(1)])
      +- AQEShuffleRead coalesced
         +- ShuffleQueryStage 0
            +- Exchange hashpartitioning(customer_id#25, 200), ENSURE_REQUIREMENTS, [plan_id=469]
               +- *(1) HashAggregate(keys=[customer_id#25], functions=[partial_count(1)])
                  +- *(1) ColumnarToRow
                     +- FileScan parquet [customer_id#25] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<customer_id:int>
+- == Initial Plan ==
   HashAggregate(keys=[customer_id#25], functions=[count(1)])
   +- Exchange hashpartitioning(customer_id#25, 200), ENSURE_REQUIREMENTS, [plan_id=455]
      +- HashAggregate(keys=[customer_id#25], functions=[partial_count(1)])
         +- FileScan parquet [customer_id#25] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<customer_id:int>
```

</details>

<details>
<summary><b>A8</b> &nbsp;Keep AQE on, set spark.sql.shuffle.partitions to 2000, build q again and collect it. How many tasks does the ...</summary>

**Still 1.** `spark.sql.shuffle.partitions` is the number of partitions a shuffle starts with. Without AQE it is exactly the number of tasks after every shuffle (Lab 01: 200 tasks for 3 statuses). With AQE it is only the upper limit: AQE merges small partitions until they reach about `spark.sql.adaptive.advisoryPartitionSizeInBytes` (64 MB). A common rule: set it high enough for your biggest shuffle and let AQE shrink it.

```python
spark.conf.set("spark.sql.shuffle.partitions", 2000)
q = orders.groupBy("customer_id").count()
q.collect()
q.explain()   # AQEShuffleRead coalesced
spark.conf.set("spark.sql.shuffle.partitions", 200)
```

Result:

```
AdaptiveSparkPlan isFinalPlan=true
      +- AQEShuffleRead coalesced
            +- Exchange hashpartitioning(customer_id#25, 2000), ENSURE_REQUIREMENTS, [plan_id=511]
   +- Exchange hashpartitioning(customer_id#25, 2000), ENSURE_REQUIREMENTS, [plan_id=497]
```

</details>

<details>
<summary><b>B1</b> &nbsp;Number the orders of customers 1 and 2 by date: show customer_id, order_id, order_date and order_no (1 = the ...</summary>

```python
w = Window.partitionBy("customer_id").orderBy("order_date", "order_id")
(orders.filter(F.col("customer_id").isin(1, 2))
       .withColumn("order_no", F.row_number().over(w))
       .select("customer_id", "order_id", "order_date", "order_no")
       .orderBy("customer_id", "order_no")
       .show())
```

</details>

<details>
<summary><b>B2</b> &nbsp;Rank the products by price inside each category, most expensive first. Show category, product_name, price and ...</summary>

```python
w = Window.partitionBy("category").orderBy(F.col("price").desc())
(products.withColumn("price_rank", F.rank().over(w))
         .select("category", "product_name", "price", "price_rank")
         .orderBy("category", "price_rank")
         .show())
```

</details>

<details>
<summary><b>B3</b> &nbsp;Show the most expensive product of each category: category, product_name and price.</summary>

```python
w = Window.partitionBy("category").orderBy(F.col("price").desc())
(products.withColumn("rn", F.row_number().over(w))
         .filter(F.col("rn") == 1)
         .select("category", "product_name", "price")
         .orderBy("category")
         .show())
```

</details>

<details>
<summary><b>B4</b> &nbsp;For every order of customers 1 and 3, show how many days passed since the same customer's previous order ...</summary>

```python
w = Window.partitionBy("customer_id").orderBy("order_date", "order_id")
(orders.filter(F.col("customer_id").isin(1, 3))
       .withColumn("days_since_prev", F.datediff("order_date", F.lag("order_date").over(w)))
       .select("customer_id", "order_id", "order_date", "days_since_prev")
       .orderBy("customer_id", "order_date", "order_id")
       .show())
```

</details>

<details>
<summary><b>B5</b> &nbsp;For the orders of customers 2 and 5, show the date of the same customer's next order (next_order_date, empty ...</summary>

```python
w = Window.partitionBy("customer_id").orderBy("order_date", "order_id")
(orders.filter(F.col("customer_id").isin(2, 5))
       .withColumn("next_order_date", F.lead("order_date").over(w))
       .select("customer_id", "order_id", "order_date", "next_order_date")
       .orderBy("customer_id", "order_date")
       .show())
```

</details>

<details>
<summary><b>B6</b> &nbsp;Running total: for customers 1 and 2, show customer_id, order_id, order_date, quantity and items_so_far (the ...</summary>

```python
w = (Window.partitionBy("customer_id").orderBy("order_date", "order_id")
           .rowsBetween(Window.unboundedPreceding, Window.currentRow))
(orders.filter(F.col("customer_id").isin(1, 2))
       .withColumn("items_so_far", F.sum("quantity").over(w))
       .select("customer_id", "order_id", "order_date", "quantity", "items_so_far")
       .orderBy("customer_id", "order_date", "order_id")
       .show())
```

</details>

<details>
<summary><b>B7</b> &nbsp;For customers 1 and 3, show each order's share of the customer's items: customer_id, order_id, quantity and ...</summary>

```python
w = Window.partitionBy("customer_id")
(orders.filter(F.col("customer_id").isin(1, 3))
       .withColumn("share_pct", F.round(F.col("quantity") / F.sum("quantity").over(w) * 100, 1))
       .select("customer_id", "order_id", "quantity", "share_pct")
       .orderBy("customer_id", "order_id")
       .show())
```

</details>

<details>
<summary><b>B8</b> &nbsp;Show the revenue (quantity × price) of delivered orders per month, the previous month's revenue and the ...</summary>

```python
monthly = (orders.filter(F.col("status") == "delivered")
                 .join(products, "product_id")
                 .groupBy(F.month("order_date").alias("month"))
                 .agg(F.sum(F.col("quantity") * F.col("price")).alias("revenue")))
w = Window.orderBy("month")
(monthly.withColumn("prev_revenue", F.lag("revenue").over(w))
        .withColumn("change", F.col("revenue") - F.col("prev_revenue"))
        .orderBy("month")
        .show())
```

A window without `partitionBy` moves all rows into one partition. That is fine for 3 monthly rows, but not for millions of rows.

</details>

<details>
<summary><b>B9</b> &nbsp;Rank the customers by their number of orders. Customers with the same number share a rank, and no rank is ...</summary>

```python
counts = orders.groupBy("customer_id").agg(F.count("*").alias("orders"))
w = Window.orderBy(F.col("orders").desc())
(counts.join(customers, "customer_id")
       .withColumn("rank", F.dense_rank().over(w))
       .select("name", "orders", "rank")
       .orderBy("rank", "name")
       .show())
```

</details>

<details>
<summary><b>B10</b> &nbsp;For each customer, show the name of the product of their first order (the earliest date; on the same day, the ...</summary>

```python
w = Window.partitionBy("customer_id").orderBy("order_date", "order_id")
first_orders = orders.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1)
(first_orders.join(customers, "customer_id")
             .join(products, "product_id")
             .select("name", F.col("product_name").alias("first_product"))
             .orderBy("name")
             .show())
```

</details>

<details>
<summary><b>B11</b> &nbsp;For customers 1 and 3, show the moving average of quantity over the current and the previous order ...</summary>

```python
w = Window.partitionBy("customer_id").orderBy("order_date", "order_id").rowsBetween(-1, Window.currentRow)
(orders.filter(F.col("customer_id").isin(1, 3))
       .withColumn("moving_avg", F.round(F.avg("quantity").over(w), 2))
       .select("customer_id", "order_id", "quantity", "moving_avg")
       .orderBy("customer_id", "order_date", "order_id")
       .show())
```

</details>

<details>
<summary><b>B12</b> &nbsp;Show the orders whose quantity is above the average quantity of the same customer's orders. Columns: ...</summary>

```python
w = Window.partitionBy("customer_id")
(orders.withColumn("avg_quantity", F.round(F.avg("quantity").over(w), 2))
       .filter(F.col("quantity") > F.col("avg_quantity"))
       .select("customer_id", "order_id", "quantity", "avg_quantity")
       .orderBy("order_id")
       .show())
```

</details>

<details>
<summary><b>B13</b> &nbsp;Show the top 2 products of each category by items delivered (ties: the smaller product_id first). Columns: ...</summary>

```python
items = (orders.filter(F.col("status") == "delivered")
               .join(products, "product_id")
               .groupBy("category", "product_id", "product_name")
               .agg(F.sum("quantity").alias("items")))
w = Window.partitionBy("category").orderBy(F.col("items").desc(), "product_id")
(items.withColumn("rank", F.row_number().over(w))
      .filter(F.col("rank") <= 2)
      .select("category", "product_name", "items", "rank")
      .orderBy("category", "rank")
      .show())
```

</details>